### Construct Binary Tree from Preorder and Inorder Traversal

Given two integer arrays preorder and inorder where preorder is the preorder traversal of a binary tree and inorder is the inorder traversal of the same tree, construct and return the binary tree.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

In [ ]:
class Solution:
    def buildTree(self, preorder: list[int], inorder: list[int]) -> TreeNode | None:

        # if base case, lists are empty
        if not preorder:
            return None

        # otherwise the first element of preorder is root
        root = TreeNode(preorder[0])
        partition = inorder.index(preorder[0])
        # the left subtree
        # all inorder elements before partition will be left subtree
        # all preorder elements from index 1 to index partion (including) will be left subtree
        root.left = self.buildTree(preorder[1:partition+1], inorder[:partition])

        # the right subtree
        # all inorder elements after the partition will be the right subtree
        # all preorder elements from partition+1 to the end will be left subtree
        root.right = self.buildTree(preorder[partition+1:], inorder[partition+1:])
        return root

### Time and Space Complexity

For the above version, the time complexity is

O(n²) worst case, O(n log n) on a balanced tree.

Each call does more than recurse. If its lists have k elements, then inorder.index(...) scans up to k elements, and the four slices copy about k elements in total. So each call costs O(k), not O(1).

The total depends on the tree's shape. Take a fully left-skewed tree, where every node has only a left child:

preorder = [1, 2, 3, 4]
inorder  = [4, 3, 2, 1]


The calls receive lists of size 4, then 3, then 2, then 1. The work is roughly 4 + 3 + 2 + 1, which for n nodes is n + (n−1) + … + 1 ≈ n²/2, so O(n²). 

On a balanced tree the lists halve at each level. 

Each level copies about n elements in total, and there are about log n levels, so it's O(n log n).

Space: O(n²) worst case, O(n) on a balanced tree.

The slices passed into a call stay alive until that call finishes. On the skewed tree, the lists of size 4, 3, 2, and 1 are all alive at once on the call stack, so the memory is about n + (n−1) + … + 1, which is O(n²). 

On a balanced tree the sizes are n, n/2, n/4, …, which adds up to about 2n, so O(n). T

he tree you build is O(n) either way.

### Optimized Version

The optimized version is the same algorithm as yours: take the root from preorder, find it in inorder, and recurse on the left part and the right part. Only two things change.

1. Build a dictionary once, so you don't search. Your version calls inorder.index(...) every time, which scans the list. Instead, make a dictionary at the start that maps each value to its position in inorder. After that, finding the root's position is one lookup.

2. Don't copy sub-lists. Pass numbers that say which part of the original lists to use. Your version slices to make smaller lists, and slicing copies. Instead, keep the two original lists and pass three numbers:

pre_start: the index in preorder of this call's root.

in_lo and in_hi: the part of inorder this call covers (in_hi isn't included).


Inside each call:

Root = preorder[pre_start].

Root's position in inorder = mid, from the dictionary.

Number of nodes on the left = mid - in_lo.

Left call: pre_start + 1, in_lo, mid. The left root is the next name in preorder, and the window ends at the root.

Right call: pre_start + 1 + left_size, mid + 1, in_hi. The right root comes after the whole left group in preorder, and the window starts after the root.

If in_lo >= in_hi, there are no nodes, so return None.

In [2]:
def construct_binary_tree(preorder, inorder):
    idx = {v: i for i, v in enumerate(inorder)}

    def build(pre_start, in_lo, in_hi):
        if in_lo >= in_hi:
            return None
        root = TreeNode(preorder[pre_start])
        mid = idx[root.val]
        left_size = mid - in_lo
        root.left  = build(pre_start + 1, in_lo, mid)
        root.right = build(pre_start + 1 + left_size, mid + 1, in_hi)
        return root

    return build(0, 0, len(inorder))